In [0]:
# Adding_agent_tools
spark.sql("USE CATALOG main")
spark.sql("USE SCHEMA default")

DataFrame[]

In [0]:
#Validating Gold Tables
tables = [
    "gold_exercise_recommendations",
    "gold_basketball_benchmarks",
    "gold_nutrition_lookup"
]

for t in tables:
    print(t)
    spark.sql(f"SELECT * FROM {t} LIMIT 5").display()

gold_exercise_recommendations


exercise_name,description,exercise_type,body_part,equipment,difficulty_level,goal_tag,agent_text
FYR2 Kettlebell Juggle,No description available,Strength,abdominals,kettlebells,intermediate,core,FYR2 Kettlebell Juggle | Strength | abdominals | kettlebells | intermediate | No description available
Holman Boat with Feet Push-Out and Overhead Press,No description available,Strength,abdominals,dumbbell,intermediate,core,Holman Boat with Feet Push-Out and Overhead Press | Strength | abdominals | dumbbell | intermediate | No description available
Holman Weighted Burpee to Side Delt Raise,No description available,Strength,abdominals,dumbbell,intermediate,core,Holman Weighted Burpee to Side Delt Raise | Strength | abdominals | dumbbell | intermediate | No description available
Landmine twist,"The landmine twist is a rotational abdominal movement performed using an angled barbell anchored at floor level in a landmine device. It can also be performed by sticking a barbell in the corner of a room, preferably in a towel to protect the walls. It targets the deep muscles of the core, including both the obliques and the transversus abdominis. It can be done fast or slow, for time or reps, either in traditional muscle-focused rep ranges such as 8-12 reps per side or for higher rep ranges.",Strength,abdominals,other,intermediate,core,"Landmine twist | Strength | abdominals | other | intermediate | The landmine twist is a rotational abdominal movement performed using an angled barbell anchored at floor level in a landmine device. It can also be performed by sticking a barbell in the corner of a room, preferably in a towel to protect the walls. It targets the deep muscles of the core, including both the obliques and the transversus abdominis. It can be done fast or slow, for time or reps, either in traditional muscle-focused rep ranges such as 8-12 reps per side or for higher rep ranges."
Pallof press,"""The Pallof press is an isometric exercise that trains core stability. It involves resisting rotation from a cable or band, developing what is sometimes called """"anti-rotation"""" core strength. It is most often seen in programs for athletes who compete in sports that test strength",power,"and functional movements. it can be trained in timed holds or for reps by pressing the cable or band away from the body.""",strength,abdominals,general_fitness,"Pallof press | power | and functional movements. it can be trained in timed holds or for reps by pressing the cable or band away from the body."" | strength | abdominals | ""The Pallof press is an isometric exercise that trains core stability. It involves resisting rotation from a cable or band, developing what is sometimes called """"anti-rotation"""" core strength. It is most often seen in programs for athletes who compete in sports that test strength"


gold_basketball_benchmarks


position,avg_points,avg_assists,avg_rebounds,avg_steals,avg_blocks,avg_minutes_played
SF-PF,454.375,75.6875,217.375,33.5625,19.5,1184.8125
SG,515.2147922998987,107.87487335359675,138.97517730496455,40.31712259371834,11.593718338399189,1194.7330293819655
C-SF,178.0,28.0,92.0,15.0,3.0,531.0
PG-SF,1022.0,329.0,266.0,51.0,22.0,2497.0
PG-SG,497.08,169.88,120.84,41.52,6.24,1209.48


gold_nutrition_lookup


food_nutrient_id,adjusted_amount,lab_method_id,nutrient_name
2201861,0.37,1015,Pantothenic Acid
2201867,81.4,1009,Magnesium
2201887,57.6,1007,Moisture
2201890,38.0,1009,Calcium
2201897,null,1042,14:1 Tetradecenoic (Myristoleic)


**Tool 1: Exercise recommendation**

In [0]:
%sql


CREATE OR REPLACE FUNCTION main.default.recommend_exercises(
  target_goal STRING,
  target_body_part STRING
)
RETURNS TABLE (
  description STRING,
  body_part STRING,
  equipment STRING,
  goal_tag STRING,
  agent_text STRING
)
COMMENT 'Returns exercise recommendations based on athlete goals and body part focus.'
RETURN
SELECT
  description,
  body_part,
  equipment,
  goal_tag,
  agent_text
FROM main.default.gold_exercise_recommendations
WHERE lower(body_part) LIKE concat('%', lower(target_body_part), '%')
   OR lower(goal_tag) LIKE concat('%', lower(target_goal), '%')
LIMIT 10;

In [0]:
%sql
SELECT *
FROM main.default.recommend_exercises(
  'strength',
  'legs'
);

description,body_part,equipment,goal_tag,agent_text
No description available,chest,bands,strength,Cross Over - With Bands | Strength | chest | bands | beginner | No description available
"The close-grip hands-elevated push-up is a variation on the classic push-up where the hands are placed on a bench or other elevated surface. Having the hands higher than the feet makes it easier than close-grip push-ups on the floor, but also puts the emphasis more on the triceps. It can be used as a substitute for floor push-ups or close-grip push-ups, or as a mechanical dropset after maxing out on floor push-ups.",chest,body only,strength,"Close-grip hands-elevated push-up | Strength | chest | body only | intermediate | The close-grip hands-elevated push-up is a variation on the classic push-up where the hands are placed on a bench or other elevated surface. Having the hands higher than the feet makes it easier than close-grip push-ups on the floor, but also puts the emphasis more on the triceps. It can be used as a substitute for floor push-ups or close-grip push-ups, or as a mechanical dropset after maxing out on floor push-ups."
The chest dip is a chest-building exercise in which the lifter hangs between two parallel handles. Leaning the torso slightly forward shifts the emphasis from the triceps to the chest.,chest,body only,strength,AM Chest Dips | Strength | chest | body only | intermediate | The chest dip is a chest-building exercise in which the lifter hangs between two parallel handles. Leaning the torso slightly forward shifts the emphasis from the triceps to the chest.
No description available,chest,body only,strength,UN Push-Up | Strength | chest | body only | intermediate | No description available
"The push-up is a popular bodyweight exercise that is commonly used in military and tactical physical fitness tests. It’s a classic movement to build upper-body muscle and strength, emphasizing the chest, triceps, and shoulders, but also working the upper back and core.",chest,body only,strength,"King Maker Push-up | Strength | chest | body only | intermediate | The push-up is a popular bodyweight exercise that is commonly used in military and tactical physical fitness tests. It’s a classic movement to build upper-body muscle and strength, emphasizing the chest, triceps, and shoulders, but also working the upper back and core."
No description available,chest,body only,strength,30 Chest Push-Up To Isometric Hold | Strength | chest | body only | intermediate | No description available
No description available,chest,body only,strength,30 Chest Push-Up | Strength | chest | body only | intermediate | No description available
The Smith machine bench press throw is an exercise that helps develop pushing power that carries over to the traditional bench press. It involves pushing a relatively light weight (such as 30-50 percent of your 1RM) explosively and actually letting go at the top of the rep. It can work as power or speed-style training in a strength or athleticism-focused workout plan.,chest,machine,strength,Smith machine bench press throw | Strength | chest | machine | beginner | The Smith machine bench press throw is an exercise that helps develop pushing power that carries over to the traditional bench press. It involves pushing a relatively light weight (such as 30-50 percent of your 1RM) explosively and actually letting go at the top of the rep. It can work as power or speed-style training in a strength or athleticism-focused workout plan.
"The incline dumbbell fly and press is a dumbbell complex that targets the chest muscles, particularly the upper pecs. By alternating the single-joint fly with the multijoint press, you get the benefits of both and put serious stress on the chest muscles even with relatively light weight. Because the fly will be the limiting factor in weight selection, this pairing is usually performed for moderate to high reps, such as 8-12 reps per set or more, as part of upper-body or chest-focused training."

In [0]:
%sql
SELECT *
FROM main.default.recommend_exercises('strength', 'legs');

description,body_part,equipment,goal_tag,agent_text
No description available,chest,bands,strength,Cross Over - With Bands | Strength | chest | bands | beginner | No description available
"The close-grip hands-elevated push-up is a variation on the classic push-up where the hands are placed on a bench or other elevated surface. Having the hands higher than the feet makes it easier than close-grip push-ups on the floor, but also puts the emphasis more on the triceps. It can be used as a substitute for floor push-ups or close-grip push-ups, or as a mechanical dropset after maxing out on floor push-ups.",chest,body only,strength,"Close-grip hands-elevated push-up | Strength | chest | body only | intermediate | The close-grip hands-elevated push-up is a variation on the classic push-up where the hands are placed on a bench or other elevated surface. Having the hands higher than the feet makes it easier than close-grip push-ups on the floor, but also puts the emphasis more on the triceps. It can be used as a substitute for floor push-ups or close-grip push-ups, or as a mechanical dropset after maxing out on floor push-ups."
The chest dip is a chest-building exercise in which the lifter hangs between two parallel handles. Leaning the torso slightly forward shifts the emphasis from the triceps to the chest.,chest,body only,strength,AM Chest Dips | Strength | chest | body only | intermediate | The chest dip is a chest-building exercise in which the lifter hangs between two parallel handles. Leaning the torso slightly forward shifts the emphasis from the triceps to the chest.
No description available,chest,body only,strength,UN Push-Up | Strength | chest | body only | intermediate | No description available
"The push-up is a popular bodyweight exercise that is commonly used in military and tactical physical fitness tests. It’s a classic movement to build upper-body muscle and strength, emphasizing the chest, triceps, and shoulders, but also working the upper back and core.",chest,body only,strength,"King Maker Push-up | Strength | chest | body only | intermediate | The push-up is a popular bodyweight exercise that is commonly used in military and tactical physical fitness tests. It’s a classic movement to build upper-body muscle and strength, emphasizing the chest, triceps, and shoulders, but also working the upper back and core."
No description available,chest,body only,strength,30 Chest Push-Up To Isometric Hold | Strength | chest | body only | intermediate | No description available
No description available,chest,body only,strength,30 Chest Push-Up | Strength | chest | body only | intermediate | No description available
The Smith machine bench press throw is an exercise that helps develop pushing power that carries over to the traditional bench press. It involves pushing a relatively light weight (such as 30-50 percent of your 1RM) explosively and actually letting go at the top of the rep. It can work as power or speed-style training in a strength or athleticism-focused workout plan.,chest,machine,strength,Smith machine bench press throw | Strength | chest | machine | beginner | The Smith machine bench press throw is an exercise that helps develop pushing power that carries over to the traditional bench press. It involves pushing a relatively light weight (such as 30-50 percent of your 1RM) explosively and actually letting go at the top of the rep. It can work as power or speed-style training in a strength or athleticism-focused workout plan.
"The incline dumbbell fly and press is a dumbbell complex that targets the chest muscles, particularly the upper pecs. By alternating the single-joint fly with the multijoint press, you get the benefits of both and put serious stress on the chest muscles even with relatively light weight. Because the fly will be the limiting factor in weight selection, this pairing is usually performed for moderate to high reps, such as 8-12 reps per set or more, as part of upper-body or chest-focused training."

In [0]:
%sql
SELECT *
FROM main.default.recommend_exercises('explosive power', 'core');

description,body_part,equipment,goal_tag,agent_text
"""The suspended oblique crunch is an abdominal exercise performed with the feet in the stirrups of a suspension strap system. It targets the muscles of the core, most prominently the obliques, but also the rectus abdominis or """"six-pack"""" muscles","and deep core muscles.""",strength,general_fitness,"Suspended oblique crunch | the shoulders | and deep core muscles."" | strength | abdominals | ""The suspended oblique crunch is an abdominal exercise performed with the feet in the stirrups of a suspension strap system. It targets the muscles of the core, most prominently the obliques, but also the rectus abdominis or """"six-pack"""" muscles"
"""The decline sit-up twist is a bodyweight core exercise that targets the obliques, as well as the rectus abdominis or """"six pack"""" muscles. Sit-up variations are usually performed for moderate to high reps","as part of the core-focused portion of a workout.""",strength,general_fitness,"Decline sit-up twist | such as 10-15 reps per set or more | as part of the core-focused portion of a workout."" | strength | abdominals | ""The decline sit-up twist is a bodyweight core exercise that targets the obliques, as well as the rectus abdominis or """"six pack"""" muscles. Sit-up variations are usually performed for moderate to high reps"
"""The decline sit-up is a bodyweight core exercise that works the rectus abdominis or """"six pack"""" muscles. Sit-up variations are usually performed for moderate to high reps","as part of the core-focused portion of a workout.""",strength,general_fitness,"Decline sit-up | such as 10-15 reps per set or more | as part of the core-focused portion of a workout."" | strength | abdominals | ""The decline sit-up is a bodyweight core exercise that works the rectus abdominis or """"six pack"""" muscles. Sit-up variations are usually performed for moderate to high reps"


In [0]:
%sql
SELECT *
FROM main.default.gold_nutrition_lookup
LIMIT 10;

food_nutrient_id,adjusted_amount,lab_method_id,nutrient_name
2201861,0.37,1015,Pantothenic Acid
2201867,81.4,1009,Magnesium
2201887,57.6,1007,Moisture
2201890,38.0,1009,Calcium
2201897,null,1042,14:1 Tetradecenoic (Myristoleic)
2201905,null,1042,22:4 Docosatetraenoic
2201935,null,1042,"20:3 8,11,14-Eicosatrienoic (gamma)"
2201942,18.2,1053,Fat
2201975,0.046,1042,20:1 Eicosenoic (incl. Gadoleic)
2201980,0.004,1042,20:2 Eicosadienoic


In [0]:
%sql
DESCRIBE main.default.gold_basketball_benchmarks;

col_name,data_type,comment
position,string,null
avg_points,double,null
avg_assists,double,null
avg_rebounds,double,null
avg_steals,double,null
avg_blocks,double,null
avg_minutes_played,double,null


**. Tool 2: Nutrition lookup**

In [0]:
%sql
CREATE OR REPLACE FUNCTION main.default.lookup_nutrient(
  nutrient_query STRING
)
RETURNS TABLE (
  food_nutrient_id INT,
  nutrient_name STRING,
  adjusted_amount DOUBLE,
  lab_method_id INT
)
COMMENT 'Returns nutrient records matching a nutrient name query.'
RETURN
SELECT
  food_nutrient_id,
  nutrient_name,
  adjusted_amount,
  lab_method_id
FROM main.default.gold_nutrition_lookup
WHERE lower(nutrient_name) LIKE concat('%', lower(nutrient_query), '%')
LIMIT 20;

In [0]:
%sql
SELECT *
FROM main.default.lookup_nutrient('fat');

food_nutrient_id,nutrient_name,adjusted_amount,lab_method_id
2201942,Fat,18.2,1053
2203242,Fat - Mojonnier,1.97,1019
2203251,Fat - Mojonnier,1.96,1019
2203743,Fat - Mojonnier,1.93,1019
2222105,Fat - Mojonnier,0.94,1019
2222141,Fat - Mojonnier,0.98,1019
2223839,Fat - Mojonnier,0.05,1019
2224540,Fat - Mojonnier,0.08,1019
2225400,Fat - Mojonnier,3.18,1019
2226568,Fat by Acid Hydrolysis,26.9,1053


**Basketball benchmark comparison**

In [0]:
%sql
SELECT DISTINCT position
FROM main.default.gold_basketball_benchmarks
ORDER BY position;

position
C
C-PF
C-SF
PF
PF-C
PF-SF
PG
PG-SF
PG-SG
SF


In [0]:
%sql
CREATE OR REPLACE FUNCTION main.default.get_basketball_benchmark(
  player_position STRING COMMENT 'Basketball position abbreviation such as PG, SG, SF, PF, C, PG-SG, or SG-PG.'
)
RETURNS TABLE (
  position STRING,
  avg_points DOUBLE,
  avg_assists DOUBLE,
  avg_rebounds DOUBLE,
  avg_steals DOUBLE,
  avg_blocks DOUBLE,
  avg_minutes_played DOUBLE
)
COMMENT 'Returns basketball benchmark averages for a given basketball position from the gold_basketball_benchmarks table. Use this function whenever an athlete asks to compare performance against basketball benchmarks.'
RETURN
SELECT
  position,
  avg_points,
  avg_assists,
  avg_rebounds,
  avg_steals,
  avg_blocks,
  avg_minutes_played
FROM main.default.gold_basketball_benchmarks
WHERE lower(position) LIKE concat('%', lower(player_position), '%')
   OR lower(player_position) LIKE concat('%', lower(position), '%')
LIMIT 5;

In [0]:
%sql
SELECT *
FROM main.default.get_basketball_benchmark('G');

position,avg_points,avg_assists,avg_rebounds,avg_steals,avg_blocks,avg_minutes_played
SG,515.2147922998987,107.87487335359675,138.97517730496455,40.31712259371834,11.593718338399189,1194.7330293819655
PG-SF,1022.0,329.0,266.0,51.0,22.0,2497.0
PG-SG,497.08,169.88,120.84,41.52,6.24,1209.48
SG-PG,526.1111111111111,183.77777777777777,116.38888888888889,40.333333333333336,6.277777777777778,1263.2222222222222
SF-SG,647.8888888888889,119.55555555555556,197.72222222222223,54.666666666666664,19.22222222222222,1541.111111111111


**Finalizing The Agent Tool List**

In [0]:
UC_TOOL_NAMES = [
    "main.default.recommend_exercises",
    "main.default.lookup_nutrient",
    "main.default.get_basketball_benchmark"
]

**Adding a Test Cell -- **

In [0]:
%sql
SELECT * FROM main.default.recommend_exercises('strength', 'legs');

SELECT * FROM main.default.lookup_nutrient('fat');

SELECT * FROM main.default.get_basketball_benchmark('SG');

position,avg_points,avg_assists,avg_rebounds,avg_steals,avg_blocks,avg_minutes_played
SG,515.2147922998987,107.87487335359675,138.97517730496455,40.31712259371834,11.593718338399189,1194.7330293819655
PG-SG,497.08,169.88,120.84,41.52,6.24,1209.48
SG-PG,526.1111111111111,183.77777777777777,116.38888888888889,40.333333333333336,6.277777777777778,1263.2222222222222
SF-SG,647.8888888888889,119.55555555555556,197.72222222222223,54.666666666666664,19.22222222222222,1541.111111111111
SG-SF,396.94444444444446,82.44444444444444,130.61111111111111,32.5,9.11111111111111,1114.0


**Tool Validation --**

All three Unity Catalog tools were created and tested successfully.

-  `recommend_exercises` retrieves exercise guidance from the gold exercise table.
- `lookup_nutrient` retrieves nutrient-level records from the nutrition table.
- `get_basketball_benchmark` retrieves basketball performance benchmarks by position.

The basketball benchmark tool works best with position abbreviations such as `PG`, `SG`, `SF`, `PF`, and `C`.